# 01 — Thu thập dữ liệu Yahoo Finance và EDA


**Notebook độc lập:** chỉ upload file `.ipynb` này lên Kaggle, không cần clone repository hay upload `src/`.
Trong Notebook Settings, chọn **Accelerator → GPU** cho notebook train và bật **Internet** nếu tải Yahoo/cài gói.
Chạy các cell từ trên xuống, hoặc Save Version → Run All. Outputs nằm dưới `/kaggle/working/hci/`.
Notebook chỉ dùng một GPU; nếu được cấp hai GPU thì `cuda:0` được sử dụng.

**Đổi dataset:** chọn `DATASET_PRESET` trong cell config: `yahoo`, `kaggle_apple`, `kaggle_tech`, hoặc `custom`.
`SOURCE="kaggle"` nhận URL/handle qua `KAGGLE_DATASET` và tên file qua `KAGGLE_FILE` (dùng kagglehub).
**Dữ liệu offline:** Add Input dataset, đặt `SOURCE="csv"` và `CSV_PATH` tới file.
`COLUMN_MAP` ánh xạ tên gốc → Date/High/Low/Close/Volume/Open; Open là tùy chọn và không được tạo giả.
Với CSV nhiều mã, đặt `TICKER_COLUMN`; chỉ chọn một ticker mỗi lần chạy. Không nối các mã thành một chuỗi.
CSV cần tối thiểu ngày, High, Low, Close, Volume. Chỉ dùng các cột giá đã chọn, tự tính lại chỉ báo.
Không đổi ticker mà vẫn trỏ vào CSV của ticker cũ. Yahoo dùng `auto_adjust=True` nhất quán.
Nếu Yahoo bị giới hạn truy cập, dùng CSV; không tắt xác minh HTTPS.

**Lưu ý Tech Stocks:** AAPL.csv thực tế có nhiều timestamp/ngày và thiếu Open; mô tả nguồn cho biết giá đã nội suy.
Preset mặc định dừng khi gặp nhiều dòng/ngày. Chỉ để khám phá, có thể đặt `CONFIG["DAILY_POLICY"]="last_snapshot"`
sau khi áp dụng preset để giữ dòng cuối mỗi ngày UTC, không cộng volume. Cách này không hoàn tác nội suy sẵn có.
Không xem dữ liệu đã nội suy là dữ liệu thô chống rò rỉ, và không dùng các cột chỉ báo/vĩ mô sẵn có trong thí nghiệm này.

Tham khảo: [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks),
[GPU trên Kaggle (NVIDIA)](https://docs.nvidia.com/datascience/deployment/latest/platforms/kaggle/),
[yfinance](https://ranaroussi.github.io/yfinance/).
[kagglehub](https://github.com/Kaggle/kagglehub),
[Apple dataset](https://www.kaggle.com/datasets/tarunpaparaju/apple-aapl-historical-stock-data),
[Tech Stocks dataset](https://www.kaggle.com/datasets/sinankr/tech-stocks-dataset).

## 1. Cấu hình — sửa cell này trước khi Run All

In [ ]:
from pathlib import Path

DATASET_PRESET = "yahoo"  # yahoo / kaggle_apple / kaggle_tech / custom
TECH_TICKER = "AAPL"      # Khi chọn kaggle_tech: AAPL / MSFT / NVDA / GOOG / TSLA

CONFIG = {
    "TICKER": "^GSPC",                 # Chỉ số S&P 500; đổi thành AAPL/MSFT cho cổ phiếu.
    "START": "2018-01-01",
    "END": "2024-01-01",               # Ngày kết thúc không được lấy vào dữ liệu.
    "SOURCE": "yfinance",              # yfinance / csv (Add Input) / kaggle (link/handle)
    "KAGGLE_DATASET": "",              # owner/dataset hoặc URL trang dataset Kaggle
    "KAGGLE_FILE": "",                 # Tên file bên trong dataset, ví dụ AAPL.csv
    "CSV_PATH": "",                    # Ví dụ /kaggle/input/my-data/GSPC_2018-01-01_2024-01-01.csv
    "CSV_PRICE_BASIS": "unknown",       # Đặt 'adjusted' nếu đã xác nhận CSV dùng giá điều chỉnh.
    "COLUMN_MAP": {},                  # Tên cột gốc -> tên chuẩn, ví dụ {"Close/Last": "Close"}
    "DATE_COLUMN": "Date", "DATE_FORMAT": None,
    "TICKER_COLUMN": None,             # Tự nhận Ticker/Symbol; đặt 'Name' nếu file nhiều mã dùng cột Name
    "CSV_SEP": ",", "CSV_ENCODING": "utf-8",
    "CSV_STRIP_TOKENS": ["$"], "CSV_THOUSANDS": ",", "CSV_DECIMAL": ".",
    "DAILY_POLICY": "strict",          # last_snapshot chỉ dành cho snapshot dữ liệu ngày, không phải nến intraday
    "DATASET_NOTE": "",
    "OUTPUT_ROOT": "/kaggle/working/hci" if Path("/kaggle/working").exists() else "kaggle_outputs",
    "TRAIN_RATIO": 0.70, "VAL_RATIO": 0.15, "TEST_RATIO": 0.15,
    "LOOKBACK": 60, "HORIZON": 1,
    "REQUIRE_GPU": True,                # Các notebook train dừng nếu chưa bật GPU.
    "SMOKE_TEST": False,                # True: chạy ngắn để kiểm tra, không dùng để kết luận.
    "SEEDS": [42, 123, 2026],
    "HIDDEN_DIM": 64, "NUM_LAYERS": 2, "DROPOUT": 0.2,
    "LEARNING_RATE": 0.001, "BATCH_SIZE": 64,
    "RUN_ANFIS": True,                 # Notebook 04: so sánh thêm nhánh Neural + Fuzzy.
    "ANFIS_NUM_RULES": 5, "ANFIS_LEARNING_RATE": 0.001,
    "EPOCHS": 60, "PATIENCE": 10,
    "SEARCH_SEED": 2026,
    "PSO_PARTICLES": 6, "PSO_ITERATIONS": 5,
    "SEARCH_EPOCHS": 8, "SEARCH_PATIENCE": 3,
    "RUN_WAVELET": False,               # Notebook 04: thêm hai mô hình cho ablation nếu True.
    "WAVELET": "db4", "WAVELET_LEVEL": 3,
    "COMMISSION": 0.001,
}

# Preset được áp dụng sau CONFIG. Dùng CONFIG.update(...) bên dưới để ghi đè preset.
PRESETS = {
    "yahoo": {}, "custom": {},
    "kaggle_apple": {
        "SOURCE": "kaggle", "TICKER": "AAPL", "START": None, "END": None,
        "KAGGLE_DATASET": "tarunpaparaju/apple-aapl-historical-stock-data",
        "KAGGLE_FILE": "HistoricalQuotes.csv", "COLUMN_MAP": {"Close/Last": "Close"},
        "DATE_FORMAT": "%m/%d/%Y", "CSV_PRICE_BASIS": "unknown",
        "DATASET_NOTE": "HistoricalQuotes.csv: kiểm tra giá điều chỉnh/split trước khi so sánh với Yahoo.",
    },
    "kaggle_tech": {
        "SOURCE": "kaggle", "TICKER": TECH_TICKER, "START": None, "END": None,
        "KAGGLE_DATASET": "sinankr/tech-stocks-dataset", "KAGGLE_FILE": f"{TECH_TICKER}.csv",
        "COLUMN_MAP": {"stock_close": "Close", "stock_high": "High", "stock_low": "Low", "stock_volume": "Volume"},
        "DATE_FORMAT": "%Y-%m-%d %H:%M:%S%z", "DAILY_POLICY": "strict",
        "DATASET_NOTE": "Nguồn đã nội suy giá và có nhiều snapshot/ngày. Mặc định strict dừng để kiểm tra. last_snapshot chỉ dùng khám phá, không chứng minh đã loại bỏ rò rỉ từ dữ liệu nguồn.",
    },
}
if DATASET_PRESET not in PRESETS:
    raise ValueError(f"Preset không hợp lệ: {DATASET_PRESET}")
CONFIG.update(PRESETS[DATASET_PRESET])
# Ví dụ sau khi Add Input: CONFIG.update(SOURCE="csv", CSV_PATH="/kaggle/input/.../HistoricalQuotes.csv")
# START=None, END=None dùng toàn bộ khoảng ngày của CSV. Có thể ghi đè bằng ngày cụ thể.

if CONFIG["SMOKE_TEST"]:
    CONFIG.update(SEEDS=[42], EPOCHS=2, PATIENCE=2, HIDDEN_DIM=16,
                  PSO_PARTICLES=2, PSO_ITERATIONS=1, SEARCH_EPOCHS=1, SEARCH_PATIENCE=1)
print(CONFIG)


## 2. Kiểm tra thư viện

In [ ]:
import importlib.util
import subprocess
import sys

# Dùng PyTorch/CUDA có sẵn của Kaggle. Chỉ cài gói phụ nếu thiếu.
required = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "torch": "torch"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(f"Thiếu thư viện nền: {missing}. Chọn image Python/GPU Kaggle hoặc cài môi trường local trước.")
optional = {}
if CONFIG.get("SOURCE") == "yfinance":
    optional["yfinance"] = "yfinance>=0.2.40"
if CONFIG.get("SOURCE") == "kaggle":
    optional["kagglehub"] = "kagglehub>=0.3.0"
if CONFIG.get("RUN_WAVELET"):
    optional["pywt"] = "PyWavelets>=1.4"
for module, package in optional.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])


## 3. Tiện ích và môi trường

In [ ]:
import hashlib
import importlib.metadata
import json
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def select_device(require_gpu=False):
    if require_gpu and not torch.cuda.is_available():
        raise RuntimeError("Chưa có GPU. Bật Accelerator GPU trên Kaggle rồi chạy lại; hoặc đặt REQUIRE_GPU=False để thử CPU.")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
    return device


def new_run(config, label):
    base = Path(config["OUTPUT_ROOT"])
    base.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    symbol = re.sub(r"[^A-Za-z0-9._-]", "_", config['TICKER'].replace('^', ''))
    root = base / f"{symbol}_{label}_{stamp}"
    root.mkdir(exist_ok=False)
    save_json(root / "config.json", config)
    versions = {}
    for name in ["numpy", "pandas", "scikit-learn", "torch", "matplotlib", "yfinance", "PyWavelets", "kagglehub"]:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    save_json(root / "environment.json", {"packages": versions, "cuda_runtime": torch.version.cuda,
              "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
    print("Output:", root)
    return root


## Tải hoặc đọc dữ liệu và kiểm tra chất lượng

In [ ]:
OHLCV = ["Open", "High", "Low", "Close", "Volume"]
REQUIRED_PRICES = ["High", "Low", "Close", "Volume"]


def kaggle_handle(value):
    value = value.strip()
    if "://" in value:
        url = urlsplit(value)
        if url.scheme != "https" or url.hostname not in {"www.kaggle.com", "kaggle.com"}:
            raise ValueError("KAGGLE_DATASET cần URL https://www.kaggle.com/... hoặc owner/dataset.")
        value = url.path.strip("/")
        if value.startswith("datasets/"):
            value = value[len("datasets/"):]
    if not re.fullmatch(r"[\w-]+/[\w-]+(?:/versions/[1-9][0-9]*)?", value):
        raise ValueError("KAGGLE_DATASET cần owner/dataset hoặc owner/dataset/versions/N.")
    return value


def csv_frame(path, config):
    # Read as strings to preserve ticker codes such as 000001 and currency formatting.
    table = pd.read_csv(path, dtype=str, sep=config.get("CSV_SEP", ","),
                        encoding=config.get("CSV_ENCODING", "utf-8"))
    table.columns = table.columns.str.strip()
    mapping = config.get("COLUMN_MAP", {})
    missing = set(mapping) - set(table.columns)
    if missing:
        raise ValueError(f"COLUMN_MAP có cột không tồn tại: {sorted(missing)}; CSV có: {list(table.columns)}")
    table = table.rename(columns=mapping)
    # Standard names are case insensitive. Other names require an explicit mapping.
    aliases = {c.casefold(): c for c in OHLCV + ["Date", "Ticker"]}
    aliases.update({"symbol": "Ticker", "datetime": "Date", "timestamp": "Date"})
    table = table.rename(columns={c: aliases.get(c.casefold(), c) for c in table.columns})
    if table.columns.duplicated().any():
        raise ValueError("Tên cột trùng sau mapping; chỉ chọn một nguồn cho mỗi cột chuẩn.")
    ticker_col = config.get("TICKER_COLUMN") or ("Ticker" if "Ticker" in table else None)
    if ticker_col:
        ticker_col = aliases.get(mapping.get(ticker_col, ticker_col).casefold(), mapping.get(ticker_col, ticker_col))
        if ticker_col not in table:
            raise ValueError(f"Không có cột ticker {ticker_col}.")
        available = table[ticker_col].dropna().str.strip().unique()
        table = table.loc[table[ticker_col].str.strip().eq(config["TICKER"])].copy()
        if table.empty:
            raise ValueError(f"Không có ticker {config['TICKER']}. Ví dụ mã hiện có: {available[:15].tolist()}")
    date_col = config.get("DATE_COLUMN", "Date")
    date_col = aliases.get(mapping.get(date_col, date_col).casefold(), mapping.get(date_col, date_col))
    if date_col not in table:
        raise ValueError(f"Không có cột ngày {date_col}. Dùng COLUMN_MAP hoặc DATE_COLUMN; CSV có {list(table.columns)}")
    if not set(REQUIRED_PRICES).issubset(table):
        raise ValueError(f"Cần tối thiểu {REQUIRED_PRICES}; Open là tùy chọn. Dùng COLUMN_MAP để đổi tên cột.")
    dates = pd.to_datetime(table[date_col].str.strip(), format=config.get("DATE_FORMAT"), errors="raise", utc=True)
    columns = [c for c in OHLCV if c in table]
    frame = table[columns].copy()
    for column in columns:
        values = frame[column].str.strip()
        for token in config.get("CSV_STRIP_TOKENS", ["$"]):
            values = values.str.replace(token, "", regex=False)
        thousands, decimal = config.get("CSV_THOUSANDS", ","), config.get("CSV_DECIMAL", ".")
        if thousands and thousands == decimal:
            raise ValueError("CSV_THOUSANDS và CSV_DECIMAL phải khác nhau.")
        if thousands:
            values = values.str.replace(thousands, "", regex=False)
        if decimal != ".":
            values = values.str.replace(decimal, ".", regex=False)
        frame[column] = pd.to_numeric(values.str.strip(), errors="raise")
    frame.index = pd.DatetimeIndex(dates).tz_convert(None)
    if frame.index.isna().any() or not np.isfinite(frame.to_numpy()).all():
        raise ValueError("Dữ liệu đã chọn chứa ngày thiếu hoặc giá/volume NaN/Inf.")
    # Daily snapshots: never sum repeated daily volume or invent a missing Open.
    frame = frame.sort_index(kind="stable")
    if frame.index.duplicated().any():
        raise ValueError("Timestamp trùng sau lọc ticker; cần xử lý nguồn dữ liệu trước.")
    policy = config.get("DAILY_POLICY", "strict")
    days = frame.index.normalize()
    extra_rows = int(days.duplicated().sum())
    if policy not in {"strict", "last_snapshot"}:
        raise ValueError("DAILY_POLICY phải là strict hoặc last_snapshot.")
    if extra_rows and policy == "strict":
        raise ValueError("Có nhiều timestamp trong một ngày. Chỉ dùng last_snapshot nếu các dòng là snapshot dữ liệu ngày; dữ liệu intraday cần tổng hợp OHLCV riêng.")
    frame.index = days
    if extra_rows:
        frame = frame.loc[~frame.index.duplicated(keep="last")].copy()
        print(f"Daily snapshots: bỏ {extra_rows} dòng lặp ngày, giữ snapshot cuối; không cộng volume.")
    info = {"column_map": mapping, "ticker_column": ticker_col, "date_column": date_col,
            "daily_policy": policy, "discarded_same_day_snapshots": extra_rows,
            "selected_columns": columns, "missing_open": "Open" not in frame}
    return frame, info


def validate_raw(frame, config):
    df = frame.copy()
    if isinstance(df.columns, pd.MultiIndex):
        if len(df.columns.get_level_values(-1).unique()) != 1:
            raise ValueError("Chỉ hỗ trợ một ticker trong một lần chạy.")
        df.columns = df.columns.get_level_values(0)
    if not set(REQUIRED_PRICES).issubset(df.columns):
        raise ValueError(f"CSV cần các cột {REQUIRED_PRICES}; Open là tùy chọn.")
    df.index = pd.to_datetime(df.index, errors="raise")
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    if df.index.isna().any() or df.index.duplicated().any():
        raise ValueError("Ngày bị thiếu hoặc trùng; kiểm tra dữ liệu gốc.")
    columns = [c for c in OHLCV if c in df]
    df = df.sort_index().loc[:, columns].apply(pd.to_numeric, errors="raise")
    if config.get("START"):
        df = df.loc[df.index >= config["START"]]
    if config.get("END"):
        df = df.loc[df.index < config["END"]]
    df = df.copy()
    if df.empty or not np.isfinite(df.to_numpy()).all():
        raise ValueError("Dữ liệu rỗng hoặc chứa NaN/Inf. Kiểm tra ngày, ticker và CSV.")
    price_cols = [c for c in columns if c != "Volume"]
    if (df[price_cols] <= 0).any().any() or (df.Volume < 0).any():
        raise ValueError("Giá phải dương, volume không âm.")
    tolerance = 1e-6 * df.Close
    if ((df.High + tolerance < df[price_cols].max(axis=1)) |
            (df.Low - tolerance > df[price_cols].min(axis=1))).any():
        raise ValueError("Quan hệ OHLC không hợp lệ.")
    df.index.name = "Date"
    return df


def load_raw(config, root):
    symbol = re.sub(r"[^A-Za-z0-9._-]", "_", config['TICKER'].replace('^', ''))
    filename = f"{symbol}_{config.get('START') or 'start'}_{config.get('END') or 'end'}.csv"
    if config["SOURCE"] in {"csv", "kaggle"}:
        if config["SOURCE"] == "kaggle":
            import kagglehub
            handle = kaggle_handle(config.get("KAGGLE_DATASET", ""))
            file = config.get("KAGGLE_FILE", "")
            if not file or file.startswith(("/", "\\")) or ".." in Path(file).parts:
                raise ValueError("Đặt KAGGLE_FILE là đường dẫn tương đối của một CSV trong dataset.")
            path = Path(kagglehub.dataset_download(handle, path=file))
        elif config.get("CSV_PATH"):
            if "://" in config["CSV_PATH"]:
                raise ValueError("CSV_PATH cần đường dẫn file; link Kaggle dùng SOURCE='kaggle' và KAGGLE_DATASET.")
            path = Path(config["CSV_PATH"])
        else:
            lookup = config.get("KAGGLE_FILE") or filename
            matches = list(Path("/kaggle/input").rglob(lookup))
            if len(matches) != 1:
                raise ValueError(f"Tìm thấy {len(matches)} file {filename}. Hãy đặt CSV_PATH chính xác trong config.")
            path = matches[0]
        frame, csv_info = csv_frame(path, config)
        source = {"source": config["SOURCE"], "path": str(path), "price_basis": config["CSV_PRICE_BASIS"],
                  "kaggle_dataset": config.get("KAGGLE_DATASET", ""), "dataset_note": config.get("DATASET_NOTE", ""), **csv_info}
    elif config["SOURCE"] == "yfinance":
        import yfinance as yf
        frame = None
        failure = None
        for attempt in range(3):
            try:
                frame = yf.download(config["TICKER"], start=config["START"], end=config["END"],
                                    auto_adjust=True, progress=False, threads=False, timeout=30)
                if frame is not None and not frame.empty:
                    break
            except Exception as exc:
                failure = str(exc)
            if attempt < 2:
                time.sleep(2 * (attempt + 1))
        if frame is None or frame.empty:
            raise RuntimeError(f"Yahoo download thất bại ({failure}). Bật Internet hoặc Add Input CSV rồi đặt SOURCE='csv'.")
        source = {"source": "yfinance", "auto_adjust": True, "price_basis": "adjusted"}
    else:
        raise ValueError("SOURCE phải là 'yfinance', 'csv' hoặc 'kaggle'.")
    df = validate_raw(frame, config)
    raw_file = root / filename
    df.to_csv(raw_file)
    source.update(ticker=config["TICKER"], rows=len(df), first_date=str(df.index[0].date()),
                  last_date=str(df.index[-1].date()), sha256=hashlib.sha256(raw_file.read_bytes()).hexdigest())
    save_json(root / "data_source.json", source)
    print("RAW:", source)
    return df


In [ ]:
RUN_DIR = new_run(CONFIG, "data")
raw = load_raw(CONFIG, RUN_DIR)
display(raw.head())


## Khám phá dữ liệu
Các hình mô tả toàn bộ lịch sử; không dùng test để chọn tham số mô hình.

In [ ]:
display(raw.describe())
print("Missing:", raw.isna().sum().to_dict(), "Duplicate dates:", raw.index.duplicated().sum())
returns = raw.Close.pct_change().dropna()
fig, axes = plt.subplots(3, 1, figsize=(13, 10))
raw.Close.plot(ax=axes[0], title=f"{CONFIG['TICKER']} — Close")
raw.Volume.plot(ax=axes[1], title="Volume")
returns.hist(bins=60, ax=axes[2])
axes[2].set_title("Phân phối lợi suất phiên")
fig.tight_layout()
fig.savefig(RUN_DIR / "eda.png", dpi=160)
plt.show()
plt.close(fig)
raw.describe().to_csv(RUN_DIR / "describe.csv")
print("CSV dùng cho các notebook khác:", list(RUN_DIR.glob("*.csv")))


### Lưu kết quả
Sau khi chạy xong, dùng **Save Version** và giữ output của phiên chạy.
Tải cả thư mục được in ở `RUN_DIR`, hoặc Add Input output này trong notebook 05.
Các lần chạy mới tạo thư mục riêng để tránh ghi đè thí nghiệm trước.